# 02 · Global integration — scVI (to run in banksy env)

Loads all per-section `_banksy_genes.h5ad` files (output of notebook 01),
concatenates them, runs QC and integrates with **scVI**
(raw counts, batch = section).

scVI is used instead of Harmony because the dataset has many batches
(3 injury models × 9 timepoints). The scVI latent embedding is used
for clustering and UMAP visualization.

**Note on BANKSY features**: scVI runs on the 500 genes only (`_banksy_genes.h5ad`),
not the 1500-feature BANKSY matrix. scVI models raw count data — the BANKSY
neighbourhood features are continuous averages, not counts, and cannot be
modelled by scVI. The 1500-feature objects are used in notebook 03 for
spatial-aware label transfer.

```
Input : BANKSY_DIR/<section>/<section>_banksy_genes.h5ad
Output: SAVE_PATH — integrated h5ad with:
          layers["counts"] | layers["normalized"] | layers["transformed"]
          obsm["X_scVI"]        <- scVI latent (20 dims)
          obsm["X_umap_scVI"]   <- UMAP of scVI latent
          obs["leiden_<res>"]   <- Leiden clusters at multiple resolutions
```

## 1 · Imports

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"

import random, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import scvi
warnings.filterwarnings("ignore")
sc.logging.print_header()
scvi.settings.seed = 0

## 2 · Paths & parameters
**Edit this cell.**

In [ ]:
BANKSY_DIR = "/path/to/banksy/output/"
SAVE_PATH  = "/path/to/output/integrated.h5ad"

PARAMS = dict(
    seed               = 1234,
    batch_key          = "section",

    # ── QC ──────────────────────────────────────────────────────────────────
    min_genes          = 17,
    max_genes          = 500,    # your panel size
    min_counts         = 5,
    max_counts         = 2500,
    min_cells_section  = 200,    # sections with fewer cells are excluded

    # ── scVI ────────────────────────────────────────────────────────────────
    n_latent           = 20,     # latent dimensions
    n_layers           = 2,      # encoder/decoder depth
    max_epochs         = 300,    # early stopping will kick in before this
    early_stopping     = True,
    n_train_cells      = 150000, # cells used for training (~1000 per section)
    batch_size         = 256,    # larger = faster on CPU

    # ── Clustering ──────────────────────────────────────────────────────────
    n_neighbors        = 30,
    leiden_resolutions = [0.3, 0.5, 0.8, 1.0, 1.5],
    umap_min_dist      = 0.001,
)

## 3 · Helper functions

### 3.1 · Environment

In [ ]:
def setup_environment(seed):
    np.random.seed(seed); random.seed(seed)
    sc.set_figure_params(facecolor="white", figsize=(8, 8))
    sc.settings.verbosity = 1
    sns.set_style("white")

### 3.2 · Load & concatenate

In [ ]:
def load_and_concatenate(banksy_dir):
    """
    Scan banksy_dir for *_banksy_genes.h5ad files (one per section).
    Concatenate into one AnnData with:
      obs['section']  = section name
      obs_names       = <section>_<barcode>  (unique across all sections)
    Sets adata.X = raw counts for scVI.
    """
    paths = []
    for section in sorted(os.listdir(banksy_dir)):
        sec_dir = os.path.join(banksy_dir, section)
        if not os.path.isdir(sec_dir):
            continue
        for f in os.listdir(sec_dir):
            if f.endswith("_banksy_genes.h5ad"):
                paths.append((section, os.path.join(sec_dir, f)))

    if not paths:
        raise FileNotFoundError(f"No *_banksy_genes.h5ad files found in {banksy_dir}")

    print(f"  Found {len(paths)} sections")

    adata_list = []
    for section, path in paths:
        ad = sc.read_h5ad(path)
        # Make obs_names unique by prepending section name
        ad.obs_names  = [f"{section}_{bc}" for bc in ad.obs_names]
        ad.obs["section"] = section
        ad.X = ad.layers["counts"].copy()   # scVI needs raw counts
        adata_list.append(ad)
        print(f"    {section:<40} {ad.n_obs:,} cells")

    adata = sc.concat(adata_list, join="inner", merge="same")
    print(f"\n  Concatenated : {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")
    print(f"  Sections     : {adata.obs['section'].nunique()}")
    print(f"  obs_names unique: {adata.obs_names.is_unique}")
    return adata

### 3.3 · Exclude sparse sections

In [ ]:
def exclude_sparse_sections(adata):
    """
    Remove sections with fewer than min_cells_section cells.
    These sections have too few cells for scVI to learn their batch effect.
    """
    min_cells = PARAMS["min_cells_section"]
    counts    = adata.obs["section"].value_counts()
    exclude   = counts[counts < min_cells].index.tolist()

    if exclude:
        print(f"  Excluding {len(exclude)} section(s) with < {min_cells} cells:")
        for s in exclude:
            print(f"    {s} — {counts[s]} cells")
        adata = adata[~adata.obs["section"].isin(exclude)].copy()
    else:
        print(f"  All sections have >= {min_cells} cells — none excluded")

    print(f"  After exclusion: {adata.n_obs:,} cells | "
          f"{adata.obs['section'].nunique()} sections")
    return adata

### 3.4 · QC filtering

In [ ]:
def run_qc(adata):
    """
    Compute QC metrics and filter low-quality cells.
    No mitochondrial filtering — MERSCOPE is a targeted panel.
    Filters on: min_genes, max_genes, min_counts, max_counts.
    """
    sc.pp.calculate_qc_metrics(
        adata, percent_top=None, log1p=False, inplace=True
    )
    print(f"  Before QC: {adata.n_obs:,} cells")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(adata.obs["n_genes_by_counts"], bins=100)
    axes[0].axvline(PARAMS["min_genes"], color="r", ls="--",
                    label=f"min={PARAMS['min_genes']}")
    axes[0].axvline(PARAMS["max_genes"], color="r", ls="--",
                    label=f"max={PARAMS['max_genes']}")
    axes[0].set_xlabel("Genes per cell")
    axes[0].set_title("Gene count distribution")
    axes[0].legend()

    axes[1].hist(adata.obs["total_counts"], bins=100)
    axes[1].axvline(PARAMS["min_counts"], color="r", ls="--",
                    label=f"min={PARAMS['min_counts']}")
    axes[1].axvline(PARAMS["max_counts"], color="r", ls="--",
                    label=f"max={PARAMS['max_counts']}")
    axes[1].set_xlabel("Total counts per cell")
    axes[1].set_title("Total count distribution")
    axes[1].legend()

    plt.suptitle("QC metrics (before filtering)")
    plt.tight_layout()
    plt.show()

    mask = (
        (adata.obs["n_genes_by_counts"] >= PARAMS["min_genes"]) &
        (adata.obs["n_genes_by_counts"] <= PARAMS["max_genes"]) &
        (adata.obs["total_counts"]      >= PARAMS["min_counts"]) &
        (adata.obs["total_counts"]      <= PARAMS["max_counts"])
    )
    adata = adata[mask].copy()
    print(f"  After QC : {adata.n_obs:,} cells kept "
          f"({adata.n_obs / mask.shape[0] * 100:.1f}% of total)")
    return adata

### 3.5 · Run scVI

In [ ]:
def run_scvi(adata):
    """
    Train scVI on a stratified subsample (faster + less memory),
    then embed all cells using the trained model.

    - Subsamples ~n_train_cells cells stratified by section
    - Runs on CPU (scVI Lightning does not support MPS)
    - Embeds ALL cells after training
    Adds obsm['X_scVI'] to adata.
    Returns the trained model.
    """
    # ── Device — CPU only (scVI Lightning does not support MPS) ─────────────
    accelerator = "cpu"
    print("  Device: CPU")

    # ── Set X = raw counts ───────────────────────────────────────────────────
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()

    # ── Stratified subsample by section ─────────────────────────────────────
    n_train   = PARAMS["n_train_cells"]
    n_per_sec = max(1, n_train // adata.obs[PARAMS["batch_key"]].nunique())
    print(f"  Subsampling: ~{n_per_sec} cells per section")

    train_pos = []
    for sec, grp in adata.obs.groupby(PARAMS["batch_key"]):
        n_sample = min(len(grp), n_per_sec)
        sampled  = grp.sample(n_sample, random_state=PARAMS["seed"])
        train_pos.extend(adata.obs_names.get_indexer(sampled.index))

    adata_train = adata[train_pos].copy()
    print(f"  Training on {adata_train.n_obs:,} cells "
          f"({adata_train.obs[PARAMS['batch_key']].nunique()} sections)...")

    # ── Train ────────────────────────────────────────────────────────────────
    scvi.model.SCVI.setup_anndata(adata_train, batch_key=PARAMS["batch_key"])
    model = scvi.model.SCVI(
        adata_train,
        n_latent = PARAMS["n_latent"],
        n_layers = PARAMS["n_layers"],
    )
    model.train(
        max_epochs     = PARAMS["max_epochs"],
        early_stopping = PARAMS["early_stopping"],
        batch_size     = PARAMS["batch_size"],
        accelerator    = accelerator,
    )

    # ── Embed ALL cells ──────────────────────────────────────────────────────
    print(f"  Embedding all {adata.n_obs:,} cells...")
    scvi.model.SCVI.setup_anndata(adata, batch_key=PARAMS["batch_key"])
    adata.obsm["X_scVI"] = model.get_latent_representation(adata)
    print(f"  Done — latent: {adata.obsm['X_scVI'].shape}")
    return model

### 3.6 · Cluster & UMAP

In [ ]:
def cluster_and_embed(adata):
    """
    Build neighbourhood graph on X_scVI, compute UMAP and Leiden clusters
    at multiple resolutions.
    Adds:
      obsm['X_umap_scVI']
      obs['leiden_<resolution>'] for each resolution in leiden_resolutions
    """
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=PARAMS["n_neighbors"])
    sc.tl.umap(adata, min_dist=PARAMS["umap_min_dist"])
    adata.obsm["X_umap_scVI"] = adata.obsm["X_umap"].copy()

    leiden_cols = []
    for r in PARAMS["leiden_resolutions"]:
        col = f"leiden_{r:g}"
        sc.tl.leiden(adata, resolution=float(r), key_added=col)
        n_cl = adata.obs[col].nunique()
        print(f"  resolution={r} → {n_cl} clusters")
        leiden_cols.append(col)

    sc.pl.umap(adata, color=leiden_cols + ["section"], ncols=3)

### 3.7 · Save

In [ ]:
def save_integrated(adata, save_path):
    """
    Restore adata.X = transformed before saving.
    All layers and embeddings are preserved.
    """
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    if "transformed" in adata.layers:
        adata.X = adata.layers["transformed"].copy()
        if sp.issparse(adata.X):
            adata.X = adata.X.tocsr()

    adata.write(save_path)
    print(f"  [✓] Saved → {save_path}")
    print(f"      {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")
    print(f"      layers : {list(adata.layers.keys())}")
    print(f"      obsm   : {list(adata.obsm.keys())}")
    print(f"      obs    : {list(adata.obs.columns)}")

## 4 · Run

In [ ]:
#Banksy kernel
setup_environment(PARAMS["seed"])

print("[1/5] Loading & concatenating sections...")
adata = load_and_concatenate(BANKSY_DIR)

In [ ]:
#Banksy kernel
print("\n[2/5] Excluding sparse sections...")
adata = exclude_sparse_sections(adata)

In [ ]:
#Banksy kernel
print("\n[3/5] QC filtering...")
adata = run_qc(adata)
adata.write("/path/to/output/file.h5ad")

In [ ]:
#scVI kernel

print("\n[4/5] scVI integration...")
model_scvi = run_scvi(adata)

In [ ]:
#Banksy kernel

print("\n[5/5] Clustering & UMAP...")
cluster_and_embed(adata)

print("\nSaving...")
save_integrated(adata, SAVE_PATH)
print("Done!")